# Can we predict 30-day readmission (`readmit_30d`)?

Plan: inspect the data, then fit models with **person-grouped cross-validation**. The data has 1,500 people with ~8 stays each, so a random row split would put the same person in both train and test, which can inflate scores through person-specific patterns. Every evaluation below keeps all of a person's stays in the same fold.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve, precision_recall_curve
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
SEED = 0
df = pd.read_csv('data.csv'); y = df.readmit_30d; groups = df.person_key
print(df.shape, 'people:', groups.nunique(), '| missing:', int(df.isna().sum().sum()), '| duplicate rows:', int(df.duplicated().sum()))
df.describe().T

## 1. Target and basic structure

In [ ]:
print('Readmission rate: %.3f  (n=%d)' % (y.mean(), len(y)))
print('Stays per person:', groups.value_counts().describe()[['min','50%','max']].to_dict())
print('Constant within person (birth_year, sex)?', (df.groupby('person_key')[['birth_year','sex']].nunique().max() == 1).to_dict())
df.groupby('visit_number').readmit_30d.agg(['mean','size']).T.round(3)

The base rate is about 20%, so accuracy is a poor metric (always predicting 'no' gets ~80%). I use ROC AUC and average precision (the no-skill level is the base rate), plus Brier score for calibration.

## 2. Univariate look at the predictors

In [ ]:
num = ['birth_year','bmi','length_of_stay_days','n_medications','lab_a','lab_b','postcode_area','visit_number']
fig, ax = plt.subplots(2, 4, figsize=(16, 6))
for a, c in zip(ax.flat, num):
    r = y.groupby(pd.qcut(df[c], 10, duplicates='drop'), observed=True).mean()
    a.plot(range(len(r)), r.values, 'o-'); a.axhline(y.mean(), color='gray', ls='--')
    a.set_title(c); a.set_xlabel('decile'); a.set_ylabel('readmit rate')
plt.tight_layout(); plt.show()
df.groupby('sex').readmit_30d.agg(['mean','size'])

### Does readmission cluster within people?

In [ ]:
pp = df.groupby('person_key').readmit_30d.agg(['mean','size'])
rng = np.random.default_rng(SEED)
sim = [rng.binomial(n, y.mean()) / n for n in pp['size']]
print('SD of per-person readmit rate: observed %.3f vs. %.3f if stays were independent' % (pp['mean'].std(), np.std(sim)))

## 3. Models, evaluated with person-grouped 5-fold CV

In [ ]:
base = ['birth_year','sex','postcode_area','bmi','length_of_stay_days','n_medications','lab_a','lab_b']
X = df[base].assign(sex=(df.sex == 'M').astype(int))
cv = GroupKFold(n_splits=5)

def make_models():
    return {
        'baseline (base rate)': DummyClassifier(strategy='prior'),
        'logistic regression': make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression(max_iter=1000)),
        'gradient boosting': HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, l2_regularization=1.0, random_state=SEED),
    }

def report(X, models):
    rows, preds = [], {}
    for n, m in models.items():
        p = cross_val_predict(m, X, y, cv=cv, groups=groups, method='predict_proba')[:, 1]
        preds[n] = p
        aucs = [roc_auc_score(y.iloc[te], p[te]) for _, te in cv.split(X, y, groups)] if n != 'baseline (base rate)' else [0.5]
        rows.append(dict(model=n, AUC=roc_auc_score(y, p), AUC_fold_sd=np.std(aucs),
                         PR_AUC=average_precision_score(y, p), Brier=brier_score_loss(y, p)))
    return pd.DataFrame(rows).set_index('model').round(4), preds

res, preds = report(X, make_models())
res

### Does the person's history help?
A stay's own outcome is unknown at prediction time, but *earlier* stays' outcomes are known. I add each person's prior readmission count/rate (strictly before the current stay, so the current label is never included) and the previous stay's length.

In [ ]:
d2 = df.sort_values(['person_key', 'visit_number']).copy()
g = d2.groupby('person_key')
d2['prior_readmits'] = g.readmit_30d.cumsum() - d2.readmit_30d
d2['prior_stays'] = g.cumcount()
d2['prior_rate'] = d2.prior_readmits / d2.prior_stays.replace(0, np.nan)
d2['prev_los'] = g.length_of_stay_days.shift()
d2 = d2.loc[df.index]
Xh = X.assign(visit_number=df.visit_number, prior_readmits=d2.prior_readmits, prior_rate=d2.prior_rate, prev_los=d2.prev_los)
mh = make_models(); del mh['baseline (base rate)']
res_h, preds_h = report(Xh, mh)
res_h

## 4. Diagnostics for the best current-stay model

In [ ]:
best = res.drop('baseline (base rate)').AUC.idxmax(); p = preds[best]
print('Best on current-stay features:', best)
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
fpr, tpr, _ = roc_curve(y, p); ax[0].plot(fpr, tpr); ax[0].plot([0, 1], [0, 1], 'k--')
ax[0].set_title('ROC (AUC=%.3f)' % roc_auc_score(y, p)); ax[0].set_xlabel('FPR'); ax[0].set_ylabel('TPR')
pr, rc, _ = precision_recall_curve(y, p); ax[1].plot(rc, pr); ax[1].axhline(y.mean(), color='k', ls='--')
ax[1].set_title('Precision-recall'); ax[1].set_xlabel('recall'); ax[1].set_ylabel('precision')
fx, fy = calibration_curve(y, p, n_bins=10, strategy='quantile'); ax[2].plot(fx, fy, 'o-'); ax[2].plot([0, .6], [0, .6], 'k--')
ax[2].set_title('Calibration'); ax[2].set_xlabel('predicted'); ax[2].set_ylabel('observed')
plt.tight_layout(); plt.show()
dec = pd.qcut(p, 10, labels=False)
pd.DataFrame({'d': dec, 'y': y}).groupby('d').y.mean().round(3).rename('observed rate by predicted-risk decile').to_frame().T

In [ ]:
lr = make_models()['logistic regression'].fit(X, y)
print(pd.Series(lr[-1].coef_[0], index=X.columns).sort_values().round(3).rename('standardised logistic coefficient'))
gb = make_models()['gradient boosting'].fit(X, y)
pi = permutation_importance(gb, X, y, scoring='roc_auc', n_repeats=10, random_state=SEED)
pd.Series(pi.importances_mean, index=X.columns).sort_values().plot.barh(title='Permutation importance (AUC drop), gradient boosting (in-sample)')
plt.show()

## 5. Practical operating point

In [ ]:
o = np.argsort(-p); rows = []
for k in [.05, .10, .20, .30]:
    sel = o[:int(k * len(p))]
    rows.append(dict(flagged=f'top {k:.0%}', precision=y.iloc[sel].mean(), recall=y.iloc[sel].sum() / y.sum(), lift=y.iloc[sel].mean() / y.mean()))
pd.DataFrame(rows).round(3)

## 6. Conclusions (figures from the run; re-running may shift the third decimal)

* **Yes, but only modestly from the current stay alone.** With person-grouped CV, logistic regression reaches AUC ≈ 0.65 (PR AUC ≈ 0.32 vs. 0.20 base rate). Gradient boosting is no better (≈ 0.63), so the signal is essentially linear and weak. (The 'baseline' row's AUC of ~0.47 is just a constant predictor on shifted folds; read it as 0.5.)
* **Drivers:** longer stay, more medications at discharge and higher `lab_a` raise risk. `bmi`, `lab_b`, `birth_year`, `sex` and `postcode_area` add essentially nothing.
* **Readmission clusters strongly within people** (per-person SD 0.27 vs. 0.15 expected if independent). Adding *prior-stay* history (count/rate of earlier readmissions) lifts AUC to ≈ 0.82 and PR AUC to ≈ 0.58. This is the largest gain, and it uses only information available before the index stay ends. **Caveat:** it helps only for returning patients; a person's first stay has no history, and there the ~0.65 model is what you have.
* **Usefulness:** the current-stay model flags the top 10% of stays with ~39% precision (about 2x lift) while catching only ~20% of readmissions. It is a rough prioritisation aid, not a reliable individual-level predictor.
* **Caveats:** this is one dataset with no temporal split (no admission dates, only `visit_number`), so I can't test drift over time. The model is calibrated only in the sense shown in the plot above. Before relying on it, confirm that every feature (especially `n_medications`, a *discharge* value) is available at the moment the prediction would be used.